# Module 11 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

## Reinforcement Learning with Value Iteration

These are the same maps from Module 1 but the "physics" of the world have changed. In Module 1, the world was deterministic. When the agent moved "south", it went "south". When it moved "east", it went "east". Now, the agent only succeeds in going where it wants to go *sometimes*. There is a probability distribution over the possible states so that when the agent moves "south", there is a small probability that it will go "east", "north", or "west" instead and have to move from there.

There are a variety of ways to handle this problem. For example, if using A\* search, if the agent finds itself off the solution, you can simply calculate a new solution from where the agent ended up. Although this sounds like a really bad idea, it has actually been shown to work really well in video games that use formal planning algorithms (which we will cover later). When these algorithms were first designed, this was unthinkable. Thank you, Moore's Law!

Another approach is to use Reinforcement Learning which covers problems where there is some kind of general uncertainty in the actions. We're going to model that uncertainty a bit unrealistically here but it'll show you how the algorithm works.

As far as RL is concerned, there are a variety of options there: model-based and model-free, Value Iteration, Q-Learning and SARSA. You are going to use Value Iteration.

## The World Representation

As before, we're going to simplify the problem by working in a grid world. The symbols that form the grid have a special meaning as they specify the type of the terrain and the cost to enter a grid cell with that type of terrain:

```
token   terrain    cost 
.       plains     1
*       forest     3
^       hills      5
~       swamp      7
x       mountains  impassible
```

When you go from a plains node to a forest node it costs 3. When you go from a forest node to a plains node, it costs 1. You can think of the grid as a big graph. Each grid cell (terrain symbol) is a node and there are edges to the north, south, east and west (except at the edges).

There are quite a few differences between A\* Search and Reinforcement Learning but one of the most salient is that A\* Search returns a plan of N steps that gets us from A to Z, for example, A->C->E->G.... Reinforcement Learning, on the other hand, returns  a *policy* that tells us the best thing to do in **every state.**

For example, the policy might say that the best thing to do in A is go to C. However, we might find ourselves in D instead. But the policy covers this possibility, it might say, D->E. Trying this action might land us in C and the policy will say, C->E, etc. At least with offline learning, everything will be learned in advance (in online learning, you can only learn by doing and so you may act according to a known but suboptimal policy).

Nevertheless, if you were asked for a "best case" plan from (0, 0) to (n-1, n-1), you could (and will) be able to read it off the policy because there is a best action for every state. You will be asked to provide this in your assignment.

We have the same costs as before. Note that we've negated them this time because RL requires negative costs and positive rewards:

In [1]:
costs = { '.': -1, '*': -3, '^': -5, '~': -7}
costs

{'.': -1, '*': -3, '^': -5, '~': -7}

and a list of offsets for `cardinal_moves`. You'll need to work this into your **actions**, A, parameter.

In [2]:
cardinal_moves = [(0,-1), (1,0), (0,1), (-1,0)]

For Value Iteration, we require knowledge of the *transition* function, as a probability distribution.

The transition function, T, for this problem is 0.70 for the desired direction, and 0.10 each for the other possible directions. That is, if the agent selects "north" then 70% of the time, it will go "north" but 10% of the time it will go "east", 10% of the time it will go "west", and 10% of the time it will go "south". If agent is at the edge of the map, it simply bounces back to the current state.

You need to implement `value_iteration()` with the following parameters:

+ world: a `List` of `List`s of terrain (this is S from S, A, T, gamma, R)
+ costs: a `Dict` of costs by terrain (this is part of R)
+ goal: A `Tuple` of (x, y) stating the goal state.
+ reward: The reward for achieving the goal state.
+ actions: a `List` of possible actions, A, as offsets.
+ gamma: the discount rate

you will return a policy: 

`{(x1, y1): action1, (x2, y2): action2, ...}`

Remember...a policy is what to do in any state for all the states. Notice how this is different than A\* search which only returns actions to take from the start to the goal. This also explains why reinforcement learning doesn't take a `start` state.

You should also define a function `pretty_print_policy( cols, rows, policy)` that takes a policy and prints it out as a grid using "^" for up, "<" for left, "v" for down and ">" for right. Use "x" for any mountain or other impassable square. Note that it doesn't need the `world` because the policy has a move for every state. However, you do need to know how big the grid is so you can pull the values out of the `Dict` that is returned.

```
vvvvvvv
vvvvvvv
vvvvvvv
>>>>>>v
^^^>>>v
^^^>>>v
^^^>>>G
```

(Note that that policy is completely made up and only illustrative of the desired output). Please print it out exactly as requested: **NO EXTRA SPACES OR LINES**.

* If everything is otherwise the same, do you think that the path from (0,0) to the goal would be the same for both A\* Search and Q-Learning?
* What do you think if you have a map that looks like:

```
><>>^
>>>>v
>>>>v
>>>>v
>>>>G
```

has this converged? Is this a "correct" policy? What are the problems with this policy as it is?


In [3]:
def read_world(filename):
    result = []
    with open(filename) as f:
        for line in f.readlines():
            if len(line) > 0:
                result.append(list(line.strip()))
    return result

---

In [4]:
from copy import deepcopy
import numpy as np

<a id="get_transition_probability"></a>
## get_transition_probability

When calculating the Q-value of a state, action pair (Q(s, a)), a probability, or transition function, is assigned for the intended action, and a minor probability of taking another possible action. This function checks if the action being evaluated matches the intended action, and returns the probability. The transition function, T, for this problem is 0.70 for the intended direction, and 0.10 each for the other possible directions.

* **action** tuple: intended action
* **other_action** Tuple: action being evaluated to calculate Q(s, a)
  
**returns** **probability** float: probability of action taking place

In [5]:
def get_transition_probability(action, other_action):
    if action == other_action: 
        return 0.7
    else: 
        return 0.1

In [6]:
assert get_transition_probability((0, -1), (0, -1)) == 0.7
assert get_transition_probability((0, -1), (0, 1)) == 0.1
assert get_transition_probability((1, 0), (1, 0)) == 0.7
assert get_transition_probability((1, 0), (0, -1)) == 0.1

<a id="value_iteration"></a>
## value_iteration

Value iteration is an iterative algorithm used in reinforcement learning to find the optimal value function for a Markov decision process (MDP). It starts with an initial guess of the value function and repeatedly updates the policy until convergence. At each iteration, the algorithm computes the value of each state by considering the immediate reward and the expected value of transitioning to neighboring states. Value iteration converges to the optimal value function, which provides the maximum expected cumulative reward for each state, given an optimal policy. This function uses value iteration to find the optimal action to reach the goal state.

* **world** List[List[str]]: the world (terrain map) for the path to be printed upon.
* **costs** Dict[str, int]: the costs for each terrain type
* **goal** Tuple[int, int]: the goal location for the path
* **reward** int: reward for reaching goal state
* **actions** List[Tuple]: possible actions
* **gamma** float: discount rate for value iteration

**returns** **policy** Dict[Tuple: Tuple]: dictionary of the outcomes of the value iteration method. The key is the position in the world, and the value of the dictionary is the action offset

In [7]:
def value_iteration(world, costs, goal, reward, actions, gamma):
    cols, rows = len(world[0]), len(world)
    V = np.zeros((rows, cols))
    
    policy = {}

    while True: 
        delta = 0
        new_V = deepcopy(V)
        for i in range(rows):
            for j in range(cols):
                if (i, j) == goal: 
                    new_V[i][j] = reward
                    continue
                if world[i][j] == 'x':  #impassable mountain
                    policy[(i, j)] = None
                    continue

                max_value = float('-inf')
                for action in actions:

                    if not (0 <= i + action[0] < rows and 0 <= j + action[1] < cols and world[i + action[0]][j + action[1]] != 'x'):
                        continue  # Skip out of bounds or impassable target cells
                    
                    expected_value = costs.get(world[i][j], 0)
                    
                    for offset in actions:
                        next_i, next_j = i + offset[0], j + offset[1]
                        prob = get_transition_probability(action, offset)
                        if (next_i, next_j) == goal:
                            expected_value += prob * reward
                        elif (0 <= next_i < rows and 0 <= next_j < cols) and (world[next_i][next_j] != 'x'):
                            expected_value += (prob * gamma * V[next_i][next_j])
                        else:
                            expected_value += (prob * gamma * V[i][j])

                    if expected_value > max_value:
                        max_value = expected_value
                        policy[(i, j)] = action
                
                new_V[i][j] = max_value
                delta = max(delta, np.abs(V[i][j] - new_V[i][j]))
                
        V = new_V
        if delta < 0.01:
            break        
    return policy

In [8]:
assert max(0, np.abs(31 - 35)) == 4

assert (0 <= -2 < 4 and 0 <= -1 < 4) == False

test_world = [['.', '.', '.'], ['.', '*', '*'], ['.', '*', '*'], ['.', '#', '*'], ['.', '*', '*'], ['.', '.', '.']]

assert costs.get(test_world[3][1], 0) == 0
assert costs.get(test_world[0][0], 0) == -1


<a id="pretty_print_policy"></a>
## pretty_print_policy

This function provides a visualization representation of the optimal actions generated by the value iteration method. 'G' represents the goal state. 'X' represents the impassable mountain terrain. The arrows, '<', '^', '>', 'v' represent the four possible actions. The optimum action for each passable space is displayed and printed as output.

* **cols** int: number of columns in the world
* **rows** List[List]: number of rows in the world
* **policy** Dict[Tuple, Tuple]: dictionary of the outcomes of the value iteration method. The key is the position in the world, and the value of the dictionary is the action offset
* **goal** Tuple[int, int]: the goal location for the path
  
**returns** None

In [9]:
def pretty_print_policy( cols, rows, policy, goal):
    direction_map = {(0,-1): '<', (1,0): 'v', (0,1): '>', (-1,0): '^', None: 'x'}
    grid = [[' ' for x in range(cols)] for y in range(rows)]
    
    for key, value in policy.items():
        i, j = key
        action = direction_map[value]
        grid[i][j] = action
        
    goal_i, goal_j = goal
    grid[goal_i][goal_j] = 'G'

    for row in grid:
        print(''.join(row))
    
    return grid

In [10]:
policy_test = {
    (0, 0): (0, -1),
    (0, 1): (0, -1),
    (0, 2): (0, 1),
    (1, 0): (0, 1),
    (1, 1): (1, 0),
    (1, 2): (1, 0),
    (2, 0): (-1, 0),
    (2, 1): None,
    (2, 2): (-1, 0)
}

grid = pretty_print_policy(3, 3, policy_test, (2, 2))

assert grid[0] == ['<', '<', '>']
assert grid[1] == ['>', 'v', 'v']        
assert grid[2] == ['^', 'x', 'G']

<<>
>vv
^xG


## Value Iteration

### Small World

In [11]:
small_world = read_world( "small.txt")

In [12]:
goal = (len(small_world)-1, len(small_world[0])-1)
gamma = 0.9
reward = 1000

small_policy = value_iteration(small_world, costs, goal, reward, cardinal_moves, gamma)

In [13]:
cols = len(small_world[0])
rows = len(small_world)

grid = pretty_print_policy(cols, rows, small_policy, goal)

v>>>vv
vv>>vv
vvv>vv
vvvxvv
v>>vvv
>>>>vv
>>>>>G


### Large World

In [14]:
large_world = read_world( "large.txt")

In [15]:
goal = (len(large_world)-1, len(large_world[0])-1) # Lower Right Corner FILL ME IN
gamma = 0.9
reward = 1000

large_policy = value_iteration(large_world, costs, goal, reward, cardinal_moves, gamma)

In [16]:
cols = len(large_world[0])
rows = len(large_world)

grid = pretty_print_policy( cols, rows, large_policy, goal)

v<<<<<v<<<v>>>>>>>>>>>>>>>v
^<<<<<<<<vvv>>>>>^xxxxxxx>v
^^^^xx^<vvv<<^>^^xxxv<<xxvv
^^^^<xxxvvv<<^>>^>>vv<<xx>v
^^^^<xx>>vv<<>>>>>>vvvxxxvv
^^^^xx>>>v<<<<>^>>>>vvvx>vv
v<<xx>>>>v<<xxx>>>>>>>>>>vv
v<<<>>>>>^<<<<xxxv^>>^>>vvv
v<<<<>>>^^^<<<xxvv>v>>>>vvv
^<<<<>>^^^^xxxx>vvvv>>>>vvv
^<<<>>>^<^xxx>>>>>vvxxx>vvv
^^<<>>v^<xx>>>^^^>vvvxx>vvv
vvvv>vv<<xx>^^^^>>>vvx>>vvv
vvvv>vv<vv>>^<<<>>>>>>>>vvv
vvv<x><<<vvv^^^>>>>>^^x>vvv
vv<xxx^<>vvvxxx>>>>^^xxvvvv
vvxx>>^>>>>>vvxxx>^xxxvvvvv
><<xx>>>>>>>>>vvxxxx>>>>vvv
^^^xxx>>>>>>>>>>vv>v>>>>vvv
^^^<xxx>>>>>>>>vvvvv>>vvvvv
^<<^vvxx>>>>>^x>>>vvvvvvvvv
^<<vvvvxxx>^xx>>>>>vvvvvvvv
^^>>vvvvvxxxx>>>>>>>>vvvvvv
^^>>>vvvvvv>>^^^^xx>>>>vvvv
^x>>vvvvvvvxxx^^xxvxx>>vvvv
vxxx>>>>>vvvxxxx>>>vxxx>>vv
>>>>>>>>>>>>>>>>>>>>>>>>>>G


## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.